# ResNet-18 on Residual Autocorrelation with Prompt-Aware AI Sampling

This notebook keeps the residual-autocorrelation representation, but changes how the data is sampled:

1. Uses the raw `train`, `validation`, and `test` folders plus their manifests
2. Builds prompt groups from contiguous manifest runs in row-index order
3. Keeps all real images from eligible prompt groups
4. Picks exactly one AI image per prompt group
5. Resamples the AI choice every training epoch so the model gradually sees all generator variants
6. Uses fixed deterministic AI selections for validation and test so evaluation is stable

This gives you prompt-level balance without permanently discarding the extra AI variants during training.


In [1]:
import os
import csv
import random
import time
from pathlib import Path

import matplotlib.pylab as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import wandb
from PIL import Image
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from torchvision.transforms import functional as TF

# Use MPS on Apple Silicon when available, otherwise fall back to CPU.
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(device)


mps


In [2]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "requirements.txt").exists() and (PROJECT_ROOT.parent / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "output"

TRAIN_MANIFEST = DATA_ROOT / "metadata" / "train_manifest.csv"
VAL_MANIFEST = DATA_ROOT / "metadata" / "validation_manifest.csv"
TEST_MANIFEST = DATA_ROOT / "metadata" / "test_manifest.csv"

WANDB_ENTITY = "william-em-watson-university-of-calgary-in-alberta"
WANDB_PROJECT = "ai-gen-image-detection"
WANDB_API_KEY = os.getenv("WANDB_API_KEY", "")


In [3]:
# Residual-autocorrelation preprocessing plus prompt-aware sampling
image_size = 224
gaussian_kernel_size = 5
gaussian_sigma = 1.0


class EnsureMinSize:
    def __init__(self, min_size=224, interpolation=transforms.InterpolationMode.BILINEAR):
        self.min_size = min_size
        self.interpolation = interpolation

    def __call__(self, image):
        width, height = image.size
        if width >= self.min_size and height >= self.min_size:
            return image

        scale = max(self.min_size / width, self.min_size / height)
        new_width = max(self.min_size, round(width * scale))
        new_height = max(self.min_size, round(height * scale))
        return TF.resize(image, (new_height, new_width), interpolation=self.interpolation)


class ResidualAutocorrelationTransform:
    def __init__(self, image_size=224, gaussian_kernel_size=5, gaussian_sigma=1.0):
        self.image_size = image_size
        self.gaussian_kernel_size = gaussian_kernel_size
        self.gaussian_sigma = gaussian_sigma
        self.ensure_min_size = EnsureMinSize(min_size=image_size)

    def __call__(self, image):
        image = self.ensure_min_size(image)
        gray_image = TF.rgb_to_grayscale(image, num_output_channels=1)
        gray_tensor = TF.to_tensor(gray_image)

        blurred = TF.gaussian_blur(
            gray_tensor,
            kernel_size=[self.gaussian_kernel_size, self.gaussian_kernel_size],
            sigma=[self.gaussian_sigma, self.gaussian_sigma],
        )
        residual = gray_tensor - blurred
        residual = residual - residual.mean()

        residual_2d = residual.squeeze(0)
        spectrum = torch.fft.fft2(residual_2d)
        autocorr = torch.fft.ifft2(spectrum * torch.conj(spectrum)).real
        autocorr = torch.fft.fftshift(autocorr, dim=(-2, -1)).unsqueeze(0)
        autocorr = TF.center_crop(autocorr, [self.image_size, self.image_size])

        autocorr = autocorr - autocorr.amin()
        autocorr = autocorr / (autocorr.amax() + 1e-6)
        autocorr = (autocorr - 0.5) / 0.5
        return autocorr


def load_prompt_groups(manifest_path):
    with manifest_path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))

    rows.sort(key=lambda row: int(row["row_index"]))
    groups = []
    start = 0

    while start < len(rows):
        caption = rows[start]["caption"]
        end = start
        while end < len(rows) and rows[end]["caption"] == caption:
            end += 1

        group_rows = rows[start:end]
        groups.append(
            {
                "caption": caption,
                "rows": group_rows,
                "real_rows": [row for row in group_rows if row["label_a_name"] == "real"],
                "ai_rows": [row for row in group_rows if row["label_a_name"] == "ai_generated"],
            }
        )
        start = end

    return groups


class PromptBalancedResidualDataset(Dataset):
    def __init__(self, manifest_path, data_root, transform, selection_mode, seed=42):
        self.data_root = Path(data_root)
        self.transform = transform
        self.selection_mode = selection_mode
        self.seed = seed

        self.classes = ["ai_generated", "real"]
        self.class_to_idx = {name: idx for idx, name in enumerate(self.classes)}
        self.prompt_groups = load_prompt_groups(Path(manifest_path))
        self.eligible_groups = [
            group for group in self.prompt_groups if group["real_rows"] and group["ai_rows"]
        ]
        self.skipped_groups = len(self.prompt_groups) - len(self.eligible_groups)
        self.records = []
        self.resample_epoch(0)

    def resample_epoch(self, epoch=0):
        rng = random.Random(self.seed + epoch)
        records = []

        for group in self.eligible_groups:
            records.extend(group["real_rows"])
            if self.selection_mode == "random_per_epoch":
                selected_ai_row = rng.choice(group["ai_rows"])
            else:
                selected_ai_row = group["ai_rows"][rng.randrange(len(group["ai_rows"]))]
            records.append(selected_ai_row)

        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        image_path = self.data_root / row["image_path"]
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)
        label = self.class_to_idx[row["label_a_name"]]
        return image, label


residual_autocorr_transform = ResidualAutocorrelationTransform(
    image_size=image_size,
    gaussian_kernel_size=gaussian_kernel_size,
    gaussian_sigma=gaussian_sigma,
)

train_dataset = PromptBalancedResidualDataset(
    manifest_path=TRAIN_MANIFEST,
    data_root=DATA_ROOT,
    transform=residual_autocorr_transform,
    selection_mode="random_per_epoch",
    seed=42,
)
val_dataset = PromptBalancedResidualDataset(
    manifest_path=VAL_MANIFEST,
    data_root=DATA_ROOT,
    transform=residual_autocorr_transform,
    selection_mode="deterministic",
    seed=123,
)
test_dataset = PromptBalancedResidualDataset(
    manifest_path=TEST_MANIFEST,
    data_root=DATA_ROOT,
    transform=residual_autocorr_transform,
    selection_mode="deterministic",
    seed=456,
)

batch_size = 16
num_workers = 0

trainloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
valloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
testloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)


In [4]:
class_names = train_dataset.classes
print(class_names)
print("Train set:", len(train_dataset))
print("Val set:", len(val_dataset))
print("Test set:", len(test_dataset))
print("Train eligible prompt groups:", len(train_dataset.eligible_groups), "| skipped:", train_dataset.skipped_groups)
print("Val eligible prompt groups:", len(val_dataset.eligible_groups), "| skipped:", val_dataset.skipped_groups)
print("Test eligible prompt groups:", len(test_dataset.eligible_groups), "| skipped:", test_dataset.skipped_groups)


['ai_generated', 'real']
Train set: 4900
Val set: 0
Test set: 0
Train eligible prompt groups: 2450 | skipped: 1
Val eligible prompt groups: 0 | skipped: 3099
Test eligible prompt groups: 0 | skipped: 3498


In [ ]:
train_iterator = iter(trainloader)
train_batch = next(train_iterator)

In [ ]:
print(train_batch[0].size())
print(train_batch[1].size())

In [ ]:
# Visualize one residual-autocorrelation map from the training set.
sample_map = train_batch[0][2].squeeze(0).cpu()
sample_map = (sample_map * 0.5) + 0.5
sample_map = sample_map.clamp(0, 1)

plt.figure(figsize=(4, 4))
plt.imshow(sample_map.numpy(), cmap='gray')
plt.title(f"{class_names[train_batch[1][2].item()]} residual autocorrelation")
plt.axis('off')
plt.show()


In [ ]:
class AiGenModel(nn.Module):
    def __init__(self, num_classes=2, use_pretrained=False, freeze_backbone=False):
        super().__init__()

        self.use_pretrained = use_pretrained
        self.freeze_backbone = freeze_backbone

        weights = ResNet18_Weights.DEFAULT if use_pretrained else None
        self.feature_extractor = models.resnet18(weights=weights)

        original_conv = self.feature_extractor.conv1
        self.feature_extractor.conv1 = nn.Conv2d(
            1,
            original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=False,
        )

        if self.use_pretrained:
            with torch.no_grad():
                self.feature_extractor.conv1.weight.copy_(original_conv.weight.mean(dim=1, keepdim=True))

        if self.use_pretrained and self.freeze_backbone:
            for name, param in self.feature_extractor.named_parameters():
                if not name.startswith('fc.'):
                    param.requires_grad = False

        in_features = self.feature_extractor.fc.in_features
        self.feature_extractor.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.feature_extractor(x)


In [ ]:
num_classes = len(class_names)
use_pretrained = False
freeze_backbone = False

# The residual-autocorrelation input is single-channel and not RGB-like, so scratch training is the safer default.
net = AiGenModel(
    num_classes=num_classes,
    use_pretrained=use_pretrained,
    freeze_backbone=freeze_backbone,
).to(device)
print(net)


In [ ]:
criterion = nn.CrossEntropyLoss()
learning_rate = 1e-3
weight_decay = 1e-4

trainable_params = [param for param in net.parameters() if param.requires_grad]
optimizer = optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

input_representation = 'grayscale_noise_residual_autocorrelation'
sampling_strategy = 'all_real_plus_one_random_ai_per_prompt_per_epoch'
eval_sampling_strategy = 'all_real_plus_one_fixed_ai_per_prompt'

if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
else:
    print('WANDB_API_KEY is not set. Export it in your environment before training if you want W&B logging.')

wandb_config = {
    'train_manifest': str(TRAIN_MANIFEST),
    'val_manifest': str(VAL_MANIFEST),
    'test_manifest': str(TEST_MANIFEST),
    'model_name': 'resnet18',
    'use_pretrained': use_pretrained,
    'freeze_backbone': freeze_backbone,
    'image_size': image_size,
    'gaussian_kernel_size': gaussian_kernel_size,
    'gaussian_sigma': gaussian_sigma,
    'input_representation': input_representation,
    'sampling_strategy': sampling_strategy,
    'eval_sampling_strategy': eval_sampling_strategy,
    'train_eligible_prompt_groups': len(train_dataset.eligible_groups),
    'val_eligible_prompt_groups': len(val_dataset.eligible_groups),
    'test_eligible_prompt_groups': len(test_dataset.eligible_groups),
    'batch_size': batch_size,
    'num_workers': num_workers,
    'learning_rate': learning_rate,
    'weight_decay': weight_decay,
    'num_classes': num_classes,
    'device': str(device),
}

wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    config=wandb_config,
)

wandb.define_metric('epoch')
wandb.define_metric('train_loss', step_metric='epoch')
wandb.define_metric('train_acc', step_metric='epoch')
wandb.define_metric('val_loss', step_metric='epoch')
wandb.define_metric('val_acc', step_metric='epoch')
wandb.define_metric('learning_rate', step_metric='epoch')
wandb.define_metric('epoch_time_sec', step_metric='epoch')
wandb.watch(net, criterion, log='all', log_freq=50)


In [ ]:
# Tune nepochs and patience depending on how quickly validation loss plateaus.
nepochs = 10
PATH = './genai_resnet18_residual_autocorr_prompt_sampler_best.pth'

wandb.config.update({'nepochs': nepochs}, allow_val_change=True)

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(nepochs):
    train_dataset.resample_epoch(epoch)

    epoch_start_time = time.perf_counter()
    net.train()
    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for inputs, labels in trainloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = running_train_loss / len(train_dataset)
    train_acc = train_correct / train_total

    net.eval()
    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in valloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = net(inputs)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = running_val_loss / len(val_dataset)
    val_acc = val_correct / val_total
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    epoch_time_sec = time.perf_counter() - epoch_start_time

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    wandb.log({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'learning_rate': current_lr,
        'epoch_time_sec': epoch_time_sec,
    })

    print(
        f'Epoch {epoch + 1}/{nepochs} | '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

    if val_loss < best_val_loss:
        print('Saving best model')
        torch.save(net.state_dict(), PATH)
        best_val_loss = val_loss
        wandb.run.summary['best_val_loss'] = best_val_loss
        wandb.run.summary['best_epoch'] = epoch + 1

print('Finished training')


In [ ]:
# Load the best model before evaluating on the test set.
net = AiGenModel(
    num_classes=len(class_names),
    use_pretrained=use_pretrained,
    freeze_backbone=freeze_backbone,
).to(device)
net.load_state_dict(torch.load(PATH, map_location=device))
net.eval()


In [ ]:
correct = 0
total = 0
test_loss = 0.0

test_start_time = time.perf_counter()
with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = net(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_inference_time_sec = time.perf_counter() - test_start_time
test_loss = test_loss / len(test_dataset)
test_accuracy = correct / total

test_metrics = {
    'test_loss': test_loss,
    'test_accuracy': test_accuracy,
    'test_accuracy_percent': 100 * test_accuracy,
    'test_inference_time_sec': test_inference_time_sec,
}
wandb.log(test_metrics)
for key, value in test_metrics.items():
    wandb.run.summary[key] = value

print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {100 * test_accuracy:.2f}%')
print(f'Test inference time: {test_inference_time_sec:.2f}s')



In [ ]:
# Optional: per-class accuracy helps us see whether one class is being ignored.
class_correct = {name: 0 for name in class_names}
class_total = {name: 0 for name in class_names}

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)

        for label, pred in zip(labels, predicted):
            class_name = class_names[label.item()]
            class_total[class_name] += 1
            if label.item() == pred.item():
                class_correct[class_name] += 1

for class_name in class_names:
    accuracy = class_correct[class_name] / max(class_total[class_name], 1)
    print(f'{class_name}: {100 * accuracy:.2f}% ({class_correct[class_name]}/{class_total[class_name]})')
    wandb.log({f'test_accuracy_{class_name}': accuracy})
    wandb.run.summary[f'test_accuracy_{class_name}'] = accuracy



In [ ]:
# Optional: visualize training curves.
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='train')
plt.plot(history['val_loss'], label='val')
plt.title('Loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='train')
plt.plot(history['val_acc'], label='val')
plt.title('Accuracy')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# W&B already turns the logged epoch metrics into charts automatically.



In [ ]:
# Optional: plot the test-set confusion matrix.
confusion_matrix = torch.zeros((len(class_names), len(class_names)), dtype=torch.int64)

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)

        for label, pred in zip(labels.view(-1), predicted.view(-1)):
            confusion_matrix[label.item(), pred.item()] += 1

cm = confusion_matrix.cpu().numpy()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
fig.colorbar(im, ax=ax)

ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Test Confusion Matrix')

threshold = cm.max() / 2 if cm.size else 0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j,
            i,
            cm[i, j],
            ha='center',
            va='center',
            color='white' if cm[i, j] > threshold else 'black',
        )

plt.tight_layout()
plt.show()

wandb.finish()
